# Multi-agent comparison: run all four configs

Open this in a `mynotebook` GPU session (PyTorch environment) and press **Run All**.

1. **Setup** checks the packages and installs any that are missing. If it then asks you to, restart the kernel (**Kernel → Restart**) and **Run All** again.
2. **Run** runs each config in `CONFIGS` and writes `outputs/comparison/<config>/`. Progress is saved after every batch, so if the session ends, **Run All** again: finished configs are skipped and an unfinished one continues where it stopped. A config that fails is reported and the next one still runs.
3. **Compare** compares every config that finished and shows the report.

The notebook can live anywhere, including your home folder for `execnotebook`: it works in `REPO`.

## 1. Setup

In [ ]:
import os
import sys

# The repo on scratch. Change this if you uploaded it somewhere else.
REPO = "/common/scratch/projectgrps/CS480/CS480G3/experiments"
# Hugging Face cache on scratch; must be set before transformers is imported.
os.environ["HF_HOME"] = "/common/scratch/projectgrps/CS480/CS480G3/huggingface_cache/"
os.environ["PYTHONUNBUFFERED"] = "1"

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("Working in", os.getcwd())

In [ ]:
import importlib
import importlib.util
import subprocess
from importlib.metadata import PackageNotFoundError, version

# Module name -> pip package name, for the packages this notebook needs beyond torch.
REQUIRED = {"peft": "peft", "omegaconf": "omegaconf", "accelerate": "accelerate",
            "bitsandbytes": "bitsandbytes", "sklearn": "scikit-learn", "pandas": "pandas", "scipy": "scipy"}


def transformers_too_old():
    try:
        major, minor = (int(part) for part in version("transformers").split(".")[:2])
    except (PackageNotFoundError, ValueError):
        return True
    return (major, minor) < (5, 16)  # Qwen3.5 needs a recent transformers, as in the qlora notebooks


def missing_packages():
    importlib.invalidate_caches()
    packages = [pip_name for module, pip_name in REQUIRED.items() if importlib.util.find_spec(module) is None]
    if transformers_too_old():
        packages.append("transformers>=5.16.1")
    return packages


to_install = missing_packages()
if to_install:
    # Install only what is missing, so packages other notebooks rely on are not upgraded.
    # Inside a virtualenv (as in mynotebook) install into it; --user only works outside one.
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
    command = [sys.executable, "-m", "pip", "install", "-q", *to_install]
    if not in_venv:
        command.insert(4, "--user")
    print("Installing:", ", ".join(to_install), "into", sys.prefix if in_venv else "the user site")
    subprocess.check_call(command)
    still_missing = missing_packages()
    if still_missing or "transformers" in sys.modules and any(p.startswith("transformers") for p in to_install):
        raise RuntimeError("Packages installed. Restart the kernel (Kernel -> Restart), then Run All again.")
print("Packages OK, transformers", version("transformers"))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Start mynotebook with 'GPU enabled notebook'.")
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {torch.cuda.get_device_name(0)}, {GPU_GB:.0f} GB")
if GPU_GB < 32:
    print("Warning: cmp_feedback_adapter loads three copies of the 9B model (about 20 GB) "
          "and may run out of memory on this GPU. It runs last, so the other configs finish first.")

## 2. Run

Remove a config from `CONFIGS` to skip it. The two grading configs take minutes; each feedback config can take an hour or more.

In [ ]:
import gc
import traceback

from omegaconf import OmegaConf
from scripts.run_stage import load_and_merge_config
from src.gfv.pipeline import run_pipeline

CONFIGS = ["cmp_grading_base", "cmp_grading_qlora", "cmp_feedback_base", "cmp_feedback_adapter"]
RUN_ROOT = "outputs/comparison"

status = {}
for name in CONFIGS:
    print(f"\n===== {name} =====", flush=True)
    output_dir = os.path.join(RUN_ROOT, name)
    try:
        cfg_obj = load_and_merge_config(f"configs/experiments/{name}.yaml")
        config = OmegaConf.to_container(cfg_obj, resolve=True)
        os.makedirs(output_dir, exist_ok=True)
        OmegaConf.save(config=cfg_obj, f=os.path.join(output_dir, "config_resolved.yaml"))
        run_pipeline(config, output_dir)
        status[name] = "finished"
    except Exception:
        traceback.print_exc()
        status[name] = "FAILED (see the error above; Run All again to resume)"
    finally:
        # Free the GPU before the next config loads its models.
        gc.collect()
        torch.cuda.empty_cache()

print("\n===== Summary =====")
for name, result in status.items():
    print(f"{name}: {result}")

## 3. Compare

Compares every config that finished (those with `metrics_by_seed.csv`) and shows `outputs/comparison/report/report.md`.

In [ ]:
from IPython.display import Markdown, display

finished = [os.path.join(RUN_ROOT, name) for name in CONFIGS
            if os.path.exists(os.path.join(RUN_ROOT, name, "metrics_by_seed.csv"))]
if not finished:
    raise RuntimeError("No config has finished yet.")
print("Comparing:", ", ".join(os.path.basename(path) for path in finished))
subprocess.check_call([sys.executable, "scripts/compare.py", *finished])
with open(os.path.join(RUN_ROOT, "report", "report.md"), encoding="utf-8") as handle:
    display(Markdown(handle.read()))